# Getting started with NI-DAQmx in Python

This notebook is a step-by-step introduction to controlling a National Instruments DAQ
device from Python with the [`nidaqmx`](https://github.com/ni/nidaqmx-python) package.
It follows the spirit of the
[SWC Python_DAQmx_examples](https://github.com/SWC-Advanced-Microscopy/Python_DAQmx_examples)
repository and builds up gradually:

1. **Analog input (AI)**: read a single value, then a finite block of samples
2. **Analog output (AO)**: write a single value, then a finite waveform
3. **AO + AI together**: play a waveform and record it at the same time, synchronised by a trigger
4. **Continuous acquisition**: stream data indefinitely, using a loop and then a callback

### What you need

* NI-DAQmx driver installed (Windows or Linux; the driver is not available on macOS)
* `pip install nidaqmx numpy matplotlib`
* A DAQ device. A **simulated device** created in NI MAX works for everything here
  (*Devices and Interfaces → Create New → Simulated NI-DAQmx Device*, e.g. a PCIe-6321).
  A simulated device returns a synthetic signal on its inputs, so it won't echo what you write to the outputs.
* **Optional loopback wiring** on real hardware: connect **AO0 → AI0** (and AO GND → AI GND).
  Then everything you write on AO0 comes back on AI0, which makes the examples easy to check.

## 0. Setup

### Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import nidaqmx
from nidaqmx.constants import AcquisitionType, TerminalConfiguration

### Find your device

Each device has a name, such as `Dev1`. It is set in NI MAX and you can change it there.
List the connected devices:

In [ ]:
system = nidaqmx.system.System.local()
print("Driver version:", system.driver_version)

for device in system.devices:
    print(f"{device.name}: {device.product_type}")
    print("   AI channels:", [c.name for c in device.ai_physical_chans])
    print("   AO channels:", [c.name for c in device.ao_physical_chans])

Set the name of the device used in the rest of the notebook:

In [ ]:
DEV = "Dev1"  # <-- change to match the output above

### The basic pattern

All `nidaqmx` code follows the same steps:

1. **Create a task**, a container for one kind of operation (e.g. AI on some channels)
2. **Add channels** to the task
3. **Configure timing** (optional; without it the task is *on-demand*, one sample per call)
4. **Read or write** data
5. **Close** the task to release the hardware

Using `with nidaqmx.Task() as task:` makes step 5 automatic. The task is closed when the
block ends, even if an error occurs. A task that is left open keeps its channels reserved,
and other tasks then fail with a "resource reserved" error.

### Why neuroscientists care about this

Almost every neuroscience rig has a DAQ board like this at its core, coordinating hardware
in time:

* **Multiphoton (two-photon) imaging.** A pulsed infrared laser is focused to a single
  point in the brain. Scan mirrors (galvanometers, and often a resonant scanner) sweep that
  point across the tissue. A photomultiplier tube (PMT) measures the fluorescence. The
  **image is built up over time**: the mirrors are driven by **AO** waveforms, the PMT is
  read on **AI**, and because they share a clock, sample *i* of the PMT is the pixel the
  mirrors were pointing at at sample *i*. ScanImage, the most widely used two-photon software,
  is built around NI DAQ hardware in exactly this way. The SWC examples this notebook is based
  on come from the same scanning-microscopy world.
* **Electrophysiology (ephys).** Voltages from electrodes are sampled at 20–30 kHz so that
  action potentials ("spikes", ~1 ms long) can be resolved. In whole-cell patch clamp, the DAQ
  sends a command waveform to the amplifier on **AO** and records the cell's response on
  **AI**. In extracellular recordings, dedicated amplifiers such as Intan or Neuropixels often
  do the digitising, and a DAQ handles stimuli, behaviour and **sync pulses** so the
  different data streams can be lined up afterwards.

Look for the **🧠 In the lab** notes at the end of each section.

---
## 1. Analog input

### 1a. Read a single value (on-demand)

The simplest possible example: no timing is configured, so `read()` returns one sample right away.

In [ ]:
with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan(f"{DEV}/ai0", min_val=-10.0, max_val=10.0)
    value = task.read()

print(f"AI0 = {value:.4f} V")

`min_val` and `max_val` set the expected input range. The driver uses them to choose the
gain, so a narrower range gives better resolution.

You can also choose the **terminal configuration**, i.e. how the input is wired:
* `RSE`: single-ended, measured against AI GND
* `DIFF`: differential, measured between a pair of pins (e.g. AI0 and AI8)
* `DEFAULT`: the device's own default, which is often differential

For the loopback wiring above, use RSE:

In [ ]:
with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0",
        terminal_config=TerminalConfiguration.RSE,
        min_val=-10.0,
        max_val=10.0,
    )
    values = [task.read() for _ in range(5)]

print(values)

On-demand reads are timed by Python, so the spacing between samples is irregular.
For proper waveforms, let the device's hardware clock do the timing.

### 1b. Finite, hardware-timed acquisition

`cfg_samp_clk_timing` tells the device to acquire `N` samples at `RATE` Hz using its
internal sample clock. `read()` then waits until all samples have arrived.

In [ ]:
RATE = 10_000      # samples per second
N_SAMPLES = 1_000  # 0.1 s of data

with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0", terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
    )
    task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.FINITE, samps_per_chan=N_SAMPLES
    )
    data = task.read(number_of_samples_per_channel=N_SAMPLES)

data = np.array(data)
t = np.arange(N_SAMPLES) / RATE

plt.plot(t * 1000, data)
plt.xlabel("Time (ms)")
plt.ylabel("AI0 (V)")
plt.title("Finite acquisition")
plt.show()

### 1c. More than one channel

Pass a channel range such as `Dev1/ai0:1`, or call `add_ai_voltage_chan` several times.
With more than one channel, `read()` returns a list of lists: one list per channel.

In [ ]:
with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0:1", terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
    )
    task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.FINITE, samps_per_chan=N_SAMPLES
    )
    data = np.array(task.read(number_of_samples_per_channel=N_SAMPLES))

print("Shape (channels, samples):", data.shape)

plt.plot(t * 1000, data.T)
plt.legend(["AI0", "AI1"])
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.show()

> **🧠 In the lab:** a finite, hardware-timed AI read is one *sweep* in patch clamp, e.g. 500 ms
> of membrane voltage after a current step. Choose the sample rate for the fastest thing you
> want to see: ≥20 kHz resolves spike shapes, while a few kHz is enough for slow signals like
> a photodiode monitoring a stimulus or the voltage from a running-wheel encoder. The input
> range matters too: an amplifier output that only swings ±1 V is measured more precisely
> with `min_val=-1, max_val=1` than with ±10 V.

---
## 2. Analog output

### 2a. Write a single value (on-demand)

Without timing, `write()` sets the output voltage immediately, and the voltage stays there
after the task closes. With the loopback wired, read it back on AI0.

In [ ]:
with nidaqmx.Task() as ao_task:
    ao_task.ao_channels.add_ao_voltage_chan(f"{DEV}/ao0", min_val=-10, max_val=10)
    ao_task.write(2.5)

# Read it back (only meaningful with AO0 -> AI0 loopback on real hardware)
with nidaqmx.Task() as ai_task:
    ai_task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0", terminal_config=TerminalConfiguration.RSE
    )
    print(f"AI0 reads {ai_task.read():.3f} V")

It's good practice to set outputs back to 0 V when you're done:

In [ ]:
def zero_ao(channel=f"{DEV}/ao0"):
    """Set an analog output channel to 0 V."""
    with nidaqmx.Task() as task:
        task.ao_channels.add_ao_voltage_chan(channel)
        task.write(0.0)


zero_ao()

### 2b. Finite, hardware-timed waveform

To play a waveform, configure a sample clock, **write the whole buffer first**
(`auto_start=False`), then start the task and wait for it to finish.
Here we play 5 cycles of a 50 Hz sine wave.

In [ ]:
AO_RATE = 10_000
FREQ = 50          # Hz
AMPLITUDE = 2.0    # V
DURATION = 0.1     # s  (5 cycles at 50 Hz)

n_ao = int(AO_RATE * DURATION)
t_ao = np.arange(n_ao) / AO_RATE
waveform = AMPLITUDE * np.sin(2 * np.pi * FREQ * t_ao)

with nidaqmx.Task() as ao_task:
    ao_task.ao_channels.add_ao_voltage_chan(f"{DEV}/ao0", min_val=-10, max_val=10)
    ao_task.timing.cfg_samp_clk_timing(
        rate=AO_RATE, sample_mode=AcquisitionType.FINITE, samps_per_chan=n_ao
    )
    ao_task.write(waveform, auto_start=False)
    ao_task.start()
    ao_task.wait_until_done(timeout=DURATION + 2)
    ao_task.stop()

zero_ao()

plt.plot(t_ao * 1000, waveform)
plt.xlabel("Time (ms)")
plt.ylabel("AO0 (V)")
plt.title("Waveform that was played")
plt.show()

> **Tip:** the last sample of a finite waveform stays on the output after the task ends.
> End the waveform on 0 V, or call `zero_ao()` afterwards, so the output isn't left at some random voltage.

> **🧠 In the lab:** on-demand AO is how you *park* things: hold a galvo mirror at the centre
> of the field of view, set a constant laser power through a Pockels cell, or set an LED
> to a fixed brightness. Finite waveforms are what you use for **stimuli**: a train of
> optogenetic light pulses, a current-step protocol in patch clamp, or a single sweep of a
> scan mirror. The note about leaving the output at the last value matters a lot here.
> A Pockels cell left at 2 V after a scan keeps sending laser power into the tissue.

---
## 3. AO and AI together

We now want to play a waveform and record at the same time. A task can't contain both
input and output channels, so we need **two tasks**. To make them start at exactly the same
moment, we use a **trigger**:

* The AI task is told to wait for the AO task's *start trigger* (`/Dev1/ao/StartTrigger`).
* We start the AI task first, so it is *armed* and waiting.
* We then start the AO task. Its start fires the trigger, and both run in lock-step.

```
ai_task.start()  ──► armed, waiting…
ao_task.start()  ──► AO starts ──(ao/StartTrigger)──► AI starts
```

Note the leading `/` in the trigger name: it is a *terminal* name, not a channel name.

In [ ]:
RATE = 10_000
DURATION = 0.2
n = int(RATE * DURATION)
t = np.arange(n) / RATE

# A test signal: a 20 Hz sine plus a step halfway through
out_wave = 1.5 * np.sin(2 * np.pi * 20 * t)
out_wave[n // 2:] += 1.0
out_wave[-1] = 0.0  # finish at 0 V

with nidaqmx.Task() as ao_task, nidaqmx.Task() as ai_task:
    # --- AO ---
    ao_task.ao_channels.add_ao_voltage_chan(f"{DEV}/ao0", min_val=-10, max_val=10)
    ao_task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.FINITE, samps_per_chan=n
    )

    # --- AI ---
    ai_task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0", terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
    )
    ai_task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.FINITE, samps_per_chan=n
    )
    ai_task.triggers.start_trigger.cfg_dig_edge_start_trig(f"/{DEV}/ao/StartTrigger")

    # --- run ---
    ao_task.write(out_wave, auto_start=False)
    ai_task.start()  # armed, waiting for the trigger
    ao_task.start()  # fires the trigger -> both start together

    in_wave = np.array(ai_task.read(number_of_samples_per_channel=n, timeout=DURATION + 2))
    ao_task.wait_until_done(timeout=DURATION + 2)

plt.plot(t * 1000, out_wave, label="AO0 (sent)")
plt.plot(t * 1000, in_wave, "--", label="AI0 (recorded)")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.legend()
plt.title("Synchronised AO → AI")
plt.show()

With loopback wiring the two traces should sit on top of each other. A simulated device
records its own synthetic signal, but the timing is still synchronised.

**Try this:** comment out the `cfg_dig_edge_start_trig` line, start the AI task *after*
the AO task, and see how the recorded trace shifts in time.

> An alternative to a shared start trigger is to have AI use AO's **sample clock**
> (`source=f"/{DEV}/ao/SampleClock"` in `cfg_samp_clk_timing`). Both tasks then run off the
> same clock edges, which matters for long acquisitions where small differences between two
> clocks add up.

> **🧠 In the lab:** this AO-triggers-AI pattern is the heart of a **scanning microscope**.
> The AO task drives the galvo mirrors with a sawtooth (fast axis) and a staircase (slow axis),
> and the AI task records the PMT. Because both start together (and, in practice, share the
> AO sample clock), PMT sample *i* belongs to the pixel at mirror position *i*, and reshaping the
> AI data into rows × columns gives you an image.
>
> In **patch clamp**, it's how a stimulus protocol stays locked to the recording: AO sends the
> voltage or current step to the amplifier, and AI records the response, sample-aligned, on every sweep.

---
## 4. Continuous acquisition

Finite tasks need to know in advance how many samples to take. For indefinite streaming,
e.g. a live monitor, use `AcquisitionType.CONTINUOUS`. The device fills a **circular buffer**
in computer memory, and your code has to read data out of it fast enough that the device
doesn't overwrite unread samples. If it falls behind, you get error -200279.

Rule of thumb, as the SWC examples recommend for USB devices: read in reasonably large
chunks, **no more than ~5–10 times per second**, rather than one sample at a time.

### 4a. Continuous AI with a simple loop

In [ ]:
RATE = 10_000
CHUNK = 1_000        # samples per read -> 10 reads per second
RUN_FOR = 3.0        # seconds

chunks = []

with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0", terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
    )
    # For continuous tasks, samps_per_chan sets the buffer size, not the total
    task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * CHUNK
    )
    task.start()

    t_start = time.time()
    while time.time() - t_start < RUN_FOR:
        chunk = task.read(number_of_samples_per_channel=CHUNK)  # blocks until CHUNK samples are ready
        chunks.append(np.array(chunk))
        print(f"\rRead {len(chunks)} chunks, mean = {chunks[-1].mean():+.3f} V", end="")

data = np.concatenate(chunks)
t = np.arange(len(data)) / RATE

plt.figure(figsize=(10, 3))
plt.plot(t, data)
plt.xlabel("Time (s)")
plt.ylabel("AI0 (V)")
plt.title(f"Continuous acquisition: {len(data)} samples")
plt.show()

### 4b. Live plot

To show the data while it is being acquired, redraw the figure after each chunk.
In Jupyter, `clear_output` gives a basic live display. It's fine for a demo; a dedicated
GUI such as pyqtgraph is better for anything serious, and the SWC repository has examples.

In [ ]:
from IPython.display import clear_output, display

RATE = 10_000
CHUNK = 2_000        # 5 updates per second
RUN_FOR = 5.0

fig, ax = plt.subplots(figsize=(10, 3))
t_chunk = np.arange(CHUNK) / RATE * 1000

with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0", terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
    )
    task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * CHUNK
    )
    task.start()

    t_start = time.time()
    while time.time() - t_start < RUN_FOR:
        chunk = task.read(number_of_samples_per_channel=CHUNK)
        ax.clear()
        ax.plot(t_chunk, chunk)
        ax.set_ylim(-10, 10)
        ax.set_xlabel("Time within chunk (ms)")
        ax.set_ylabel("AI0 (V)")
        clear_output(wait=True)
        display(fig)

plt.close(fig)

### 4c. Continuous AI with a callback

Instead of a `while` loop that blocks the notebook, you can ask DAQmx to **call a function**
each time `N` new samples are in the buffer. The callback runs in a background thread, so the
notebook stays usable while data is acquired.

The callback must take these four arguments and return `0`.

In [ ]:
RATE = 10_000
CHUNK = 2_000

collected = []


def on_samples(task_handle, every_n_samples_event_type, number_of_samples, callback_data):
    """Called by DAQmx every CHUNK samples."""
    chunk = cb_task.read(number_of_samples_per_channel=number_of_samples)
    collected.append(np.array(chunk))
    return 0


cb_task = nidaqmx.Task()
cb_task.ai_channels.add_ai_voltage_chan(
    f"{DEV}/ai0", terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
)
cb_task.timing.cfg_samp_clk_timing(
    rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * CHUNK
)
cb_task.register_every_n_samples_acquired_into_buffer_event(CHUNK, on_samples)
cb_task.start()
print("Acquiring in the background… run the next cell to stop.")

The task is running in the background, and you can run other cells meanwhile. Run the
cell below once or twice to see `collected` grow.

In [ ]:
print(f"{len(collected)} chunks collected so far")

Stop and close the task when you're done. We don't use a `with` block here because the
task has to outlive the cell it was created in, so we must call `close()` ourselves.

In [ ]:
cb_task.stop()
cb_task.close()

data = np.concatenate(collected)
plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(data)) / RATE, data)
plt.xlabel("Time (s)")
plt.ylabel("AI0 (V)")
plt.title("Acquired via callback")
plt.show()

### 4d. Continuous AO + AI

Finally, combine everything: output a waveform **continuously** while recording **continuously**.

For continuous AO, write **one period** of the waveform. By default DAQmx *regenerates* it,
replaying the buffer over and over without any further work from Python. AI is triggered
by AO's start trigger, as in section 3.

In [ ]:
RATE = 10_000
FREQ = 10            # Hz
CHUNK = 2_000        # AI read size (5 reads/s)
RUN_FOR = 3.0

# One period of a sine wave; DAQmx loops it forever
one_period = 2.0 * np.sin(2 * np.pi * FREQ * np.arange(RATE // FREQ) / RATE)

chunks = []

with nidaqmx.Task() as ao_task, nidaqmx.Task() as ai_task:
    ao_task.ao_channels.add_ao_voltage_chan(f"{DEV}/ao0", min_val=-10, max_val=10)
    ao_task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=len(one_period)
    )

    ai_task.ai_channels.add_ai_voltage_chan(
        f"{DEV}/ai0", terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
    )
    ai_task.timing.cfg_samp_clk_timing(
        rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * CHUNK
    )
    ai_task.triggers.start_trigger.cfg_dig_edge_start_trig(f"/{DEV}/ao/StartTrigger")

    ao_task.write(one_period, auto_start=False)
    ai_task.start()
    ao_task.start()

    t_start = time.time()
    while time.time() - t_start < RUN_FOR:
        chunks.append(np.array(ai_task.read(number_of_samples_per_channel=CHUNK)))

    ao_task.stop()

zero_ao()

data = np.concatenate(chunks)
t = np.arange(len(data)) / RATE
plt.figure(figsize=(10, 3))
plt.plot(t, data)
plt.xlabel("Time (s)")
plt.ylabel("AI0 (V)")
plt.title("Continuous AO (regenerated) recorded on AI0")
plt.show()

> **🧠 In the lab:** continuous acquisition is how you record for minutes or hours: an
> extracellular ephys session, a behaviour experiment logging lick sensors and wheel speed, or
> continuous imaging frame after frame. Continuous AO with regeneration is how a microscope
> repeats the same scan pattern every frame without Python having to send it again. The
> callback version is what lets acquisition software keep a live display updating and save data
> to disk while you keep using the program. If your code falls behind, error -200279 means
> **lost data**, which in an experiment can mean a lost recording session.

---
## Summary

| What | Key calls |
|---|---|
| Create / clean up | `with nidaqmx.Task() as task:` |
| Add channels | `task.ai_channels.add_ai_voltage_chan("Dev1/ai0")`, `task.ao_channels.add_ao_voltage_chan("Dev1/ao0")` |
| On-demand | no timing; `task.read()` / `task.write(v)` |
| Hardware timing | `task.timing.cfg_samp_clk_timing(rate, sample_mode=..., samps_per_chan=...)` |
| Finite output | `write(..., auto_start=False)`, `start()`, `wait_until_done()` |
| Synchronise | `ai_task.triggers.start_trigger.cfg_dig_edge_start_trig("/Dev1/ao/StartTrigger")`; start AI first |
| Continuous | `AcquisitionType.CONTINUOUS`; read in chunks in a loop or with `register_every_n_samples_acquired_into_buffer_event` |

### Common errors

* **-50103 "The specified resource is reserved"**: another task (maybe from a cell that
  crashed) still holds the channels. Close it, or restart the kernel.
* **-200279 "Attempted to read samples that are no longer available"**: continuous reads
  fell behind. Read bigger chunks or make the buffer bigger.
* **-200284 "Some or all of the samples requested have not yet been acquired"**: a read timed
  out. Check the trigger is actually firing, e.g. that the AO task was started.

### Further reading

* [SWC Python_DAQmx_examples](https://github.com/SWC-Advanced-Microscopy/Python_DAQmx_examples): more complete examples, including live plotting with pyqtgraph
* [nidaqmx-python docs](https://nidaqmx-python.readthedocs.io/) and its [examples folder](https://github.com/ni/nidaqmx-python/tree/master/examples)